# ⚛️ Módulo 5: Dinámica Molecular
## Actividad 5.1: Fundamentos de Dinámica Molecular

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_05_dinamica_molecular/01_fundamentos_dinamica_molecular.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Comprender los principios físicos que gobiernan la dinámica molecular (DM)
- Describir cómo se propagan las ecuaciones de movimiento de Newton en DM
- Identificar los componentes de una simulación: campos de fuerza, condiciones iniciales y de contorno
- Distinguir entre los distintos ensambles estadísticos (NVE, NVT, NPT)
- Implementar una simulación sencilla de partículas con Python

---

## 1. Instalación de Dependencias

In [ ]:
# Instalación de paquetes necesarios
!pip install numpy matplotlib scipy

In [ ]:
# Importación de bibliotecas
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.constants import k as k_B, N_A

print("Bibliotecas importadas correctamente")

## 2. ¿Qué es la Dinámica Molecular?

La **Dinámica Molecular (DM)** es un método de simulación computacional que sigue la evolución temporal de un sistema de partículas (átomos o moléculas) resolviendo numéricamente las ecuaciones de movimiento clásicas de Newton:

$$\mathbf{F}_i = m_i \mathbf{a}_i = m_i \frac{d^2 \mathbf{r}_i}{dt^2}$$

donde:
- $\mathbf{F}_i$ es la fuerza neta sobre el átomo $i$
- $m_i$ es su masa
- $\mathbf{r}_i$ es su posición
- La fuerza se obtiene como gradiente negativo de la energía potencial: $\mathbf{F}_i = -\nabla_i U$

### Flujo de trabajo de una simulación DM

```
Posiciones y velocidades iniciales
            ↓
  Calcular fuerzas (campo de fuerza)
            ↓
  Integrar ecuaciones de movimiento
            ↓
  Actualizar posiciones y velocidades
            ↓
   Guardar trayectoria (cada N pasos)
            ↓
    ¿Se alcanzó el tiempo final?
      No → volver a calcular fuerzas
      Sí → Analizar trayectoria
```

## 3. El Potencial de Lennard-Jones

Uno de los potenciales más utilizados para describir interacciones entre átomos es el potencial de Lennard-Jones (LJ):

$$U_{LJ}(r) = 4\varepsilon \left[ \left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6} \right]$$

- El término $r^{-12}$ modela la **repulsión de corto alcance** (Pauli)
- El término $r^{-6}$ modela la **atracción de largo alcance** (dispersión de London)
- $\varepsilon$: profundidad del pozo de energía
- $\sigma$: distancia al cruce por cero

In [ ]:
def potencial_lj(r, epsilon=1.0, sigma=1.0):
    """
    Calcula el potencial de Lennard-Jones.
    
    Args:
        r: distancia entre partículas
        epsilon: profundidad del pozo (kcal/mol)
        sigma: diámetro de colisión (Å)
    
    Returns:
        Energía potencial en las mismas unidades que epsilon
    """
    return 4 * epsilon * ((sigma / r)**12 - (sigma / r)**6)

def fuerza_lj(r, epsilon=1.0, sigma=1.0):
    """
    Calcula la fuerza derivada del potencial Lennard-Jones.
    F = -dU/dr
    """
    return 4 * epsilon * (12 * sigma**12 / r**13 - 6 * sigma**6 / r**7)

# Visualizar el potencial
r = np.linspace(0.85, 3.5, 500)
U = potencial_lj(r)
F = fuerza_lj(r)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(r, U, 'b-', linewidth=2)
axes[0].axhline(0, color='k', linestyle='--', alpha=0.5)
axes[0].axvline(2**(1/6), color='r', linestyle='--', alpha=0.7, label=f'r_min = 2^(1/6)σ ≈ {2**(1/6):.3f}')
axes[0].set_ylim(-1.5, 2)
axes[0].set_xlabel('r / σ', fontsize=12)
axes[0].set_ylabel('U(r) / ε', fontsize=12)
axes[0].set_title('Potencial de Lennard-Jones', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(r, F, 'r-', linewidth=2)
axes[1].axhline(0, color='k', linestyle='--', alpha=0.5)
axes[1].set_ylim(-5, 10)
axes[1].set_xlabel('r / σ', fontsize=12)
axes[1].set_ylabel('F(r) / (ε/σ)', fontsize=12)
axes[1].set_title('Fuerza de Lennard-Jones', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lj_potential.png', dpi=100, bbox_inches='tight')
plt.show()
print("Distancia de energía mínima r_min =", 2**(1/6), "σ")
print("Energía mínima U_min =", -1, "ε")

## 4. Ensambles Estadísticos

En DM, el **ensamble** define qué variables termodinámicas se mantienen constantes durante la simulación:

| Ensamble | Variables fijas | Descripción |
|----------|----------------|-------------|
| **NVE** | N, V, E | Microcanónico. Sin termostato ni barostato |
| **NVT** | N, V, T | Canónico. Temperatura controlada |
| **NPT** | N, P, T | Isobárico-isotérmico. Temperatura y presión controladas |
| **μVT** | μ, V, T | Gran canónico. Número de partículas variable |

- **N**: número de partículas
- **V**: volumen
- **E**: energía total
- **T**: temperatura
- **P**: presión
- **μ**: potencial químico

## 5. Temperatura y Velocidades

La temperatura en DM se relaciona con la energía cinética media de las partículas a través del **teorema de equipartición**:

$$\langle E_k \rangle = \frac{3}{2} N k_B T$$

Las velocidades iniciales se asignan siguiendo la **distribución de Maxwell-Boltzmann**:

$$f(v_x) = \sqrt{\frac{m}{2\pi k_B T}} \exp\left(-\frac{m v_x^2}{2 k_B T}\right)$$

In [ ]:
def generar_velocidades_mb(n_particulas, temperatura, masa, seed=42):
    """
    Genera velocidades iniciales según la distribución de Maxwell-Boltzmann.
    
    Args:
        n_particulas: número de partículas
        temperatura: temperatura en K
        masa: masa de cada partícula en kg
        seed: semilla para reproducibilidad
    
    Returns:
        Array de velocidades (n_particulas, 3) en m/s
    """
    np.random.seed(seed)
    # Desviación estándar de la distribución
    sigma_v = np.sqrt(k_B * temperatura / masa)
    # Velocidades en 3D
    velocidades = np.random.normal(0, sigma_v, (n_particulas, 3))
    # Eliminar momento lineal total (centro de masa estático)
    velocidades -= velocidades.mean(axis=0)
    return velocidades

# Visualizar distribución de Maxwell-Boltzmann
from scipy.stats import maxwell

temperatura = 300  # K
masa_argon = 39.948 * 1.66054e-27  # kg (argón)

velocidades = generar_velocidades_mb(10000, temperatura, masa_argon)
speeds = np.linalg.norm(velocidades, axis=1)

v_range = np.linspace(0, 1200, 300)
a_mb = np.sqrt(k_B * temperatura / masa_argon)
f_mb = maxwell.pdf(v_range, scale=a_mb)

plt.figure(figsize=(8, 5))
plt.hist(speeds, bins=80, density=True, alpha=0.6, color='steelblue', label='Muestra simulada')
plt.plot(v_range, f_mb, 'r-', linewidth=2, label='Maxwell-Boltzmann teórica')
plt.xlabel('Velocidad (m/s)', fontsize=12)
plt.ylabel('Densidad de probabilidad', fontsize=12)
plt.title(f'Distribución de Maxwell-Boltzmann - Ar a {temperatura} K', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('maxwell_boltzmann.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Simulación Mínima 1D con Lennard-Jones

Implementaremos una simulación simplificada en 1D de dos partículas con potencial LJ para ilustrar el ciclo básico de DM.

In [ ]:
def simular_dos_particulas_lj(r0=1.5, v0=0.2, dt=0.001, n_pasos=5000,
                               epsilon=1.0, sigma=1.0, m=1.0):
    """
    Simulación de dos partículas en 1D con potencial LJ usando el integrador de Verlet.
    
    Args:
        r0: separación inicial entre las dos partículas
        v0: velocidad inicial relativa
        dt: paso de tiempo
        n_pasos: número de pasos de integración
        epsilon, sigma: parámetros LJ
        m: masa de cada partícula
    
    Returns:
        tiempos, distancias, energías
    """
    # Posiciones iniciales (centro de masa en 0)
    x1 = -r0 / 2
    x2 =  r0 / 2
    # Velocidades iniciales (momento total = 0)
    v1 = -v0 / 2
    v2 =  v0 / 2

    tiempos, distancias, E_kin, E_pot = [], [], [], []

    for step in range(n_pasos):
        r = x2 - x1
        if r < 0.5 * sigma:
            break  # evitar singularidad

        # Fuerza sobre cada partícula
        f = fuerza_lj(r, epsilon, sigma)
        f1 = f   # fuerza sobre 1 apunta hacia +x
        f2 = -f  # fuerza sobre 2 apunta hacia -x

        # Integración Verlet (velocidad-Verlet)
        a1 = f1 / m
        a2 = f2 / m
        x1 += v1 * dt + 0.5 * a1 * dt**2
        x2 += v2 * dt + 0.5 * a2 * dt**2
        v1 += a1 * dt
        v2 += a2 * dt

        # Registrar
        tiempos.append(step * dt)
        distancias.append(r)
        E_kin.append(0.5 * m * (v1**2 + v2**2))
        E_pot.append(potencial_lj(r, epsilon, sigma))

    return (np.array(tiempos), np.array(distancias),
            np.array(E_kin), np.array(E_pot))

# Ejecutar simulación
t, r_arr, Ek, Ep = simular_dos_particulas_lj()
E_total = Ek + Ep

fig, axes = plt.subplots(2, 1, figsize=(10, 8))

axes[0].plot(t, r_arr, 'b-', linewidth=1)
axes[0].axhline(2**(1/6), color='r', linestyle='--', label='r_min (equilibrio)')
axes[0].set_xlabel('Tiempo (u.r.)', fontsize=12)
axes[0].set_ylabel('Distancia r / σ', fontsize=12)
axes[0].set_title('Trayectoria - Separación entre partículas', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, Ek, 'r-', label='E cinética', linewidth=1)
axes[1].plot(t, Ep, 'b-', label='E potencial', linewidth=1)
axes[1].plot(t, E_total, 'k--', label='E total', linewidth=1.5)
axes[1].set_xlabel('Tiempo (u.r.)', fontsize=12)
axes[1].set_ylabel('Energía (ε)', fontsize=12)
axes[1].set_title('Conservación de la Energía Total', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('conservacion_energia.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Desviación de energía total: {(E_total.max()-E_total.min())/abs(E_total.mean()):.2e}")

## 7. Parámetros Importantes en una Simulación DM

| Parámetro | Descripción | Valor típico |
|-----------|-------------|-------------|
| **Paso de tiempo (dt)** | Intervalo de integración | 1-2 fs |
| **Temperatura** | Temperatura del ensamble | 300 K |
| **Presión** | Presión del sistema | 1 bar |
| **Tiempo de equilibración** | Pasos antes de producción | 1-5 ns |
| **Tiempo de producción** | Tiempo de recogida de datos | 10-100 ns |
| **Cutoff electrostático** | Radio de corte de interacciones | 10-12 Å |
| **Frecuencia de guardado** | Cada cuántos pasos guardar | 5000-10000 |

## 8. Ejercicios

### Ejercicio 1 (Básico)
Modifica los parámetros `r0` y `v0` de la simulación de dos partículas y observa cómo cambia la trayectoria. ¿Qué ocurre si `r0 < sigma`?

### Ejercicio 2 (Intermedio)
Extiende la simulación a 3 partículas en 1D. Calcula la energía total del sistema y verifica su conservación.

### Ejercicio 3 (Avanzado)
Implementa la distribución de Maxwell-Boltzmann para un gas de Argón a distintas temperaturas (100, 300, 1000 K). Calcula la velocidad más probable, la media y la cuadrática media (rms) para cada temperatura y compáralas con los valores teóricos.

In [ ]:
# Ejercicio 1 - Espacio para respuesta
# Modifica los valores de r0 y v0 y observa el comportamiento

for r0_test in [0.9, 1.1, 2**(1/6), 2.0, 3.0]:
    t_test, r_test, Ek_test, Ep_test = simular_dos_particulas_lj(r0=r0_test, v0=0.1)
    E_total_test = Ek_test + Ep_test
    drift = (E_total_test.max() - E_total_test.min()) / abs(E_total_test.mean()) if len(E_total_test) > 0 else float('nan')
    print(f"r0 = {r0_test:.4f} σ | Pasos simulados: {len(t_test):4d} | Deriva energía: {drift:.2e}")

## 9. Recursos Adicionales

- **Libros:**
  - Frenkel & Smit, *Understanding Molecular Simulation* (2002)
  - Allen & Tildesley, *Computer Simulation of Liquids* (2017)
  - Leach, *Molecular Modelling: Principles and Applications* (2001)

- **Herramientas:**
  - [GROMACS](https://www.gromacs.org/) - Software de DM de alto rendimiento
  - [OpenMM](https://openmm.org/) - Biblioteca de DM en Python
  - [NAMD](https://www.ks.uiuc.edu/Research/namd/) - DM para biomoléculas
  - [AMBER](https://ambermd.org/) - Suite de DM

- **Tutoriales en línea:**
  - [GROMACS Tutorials](http://www.mdtutorials.com/gmx/)
  - [OpenMM User Guide](https://openmm.org/documentation)
  - [MDAnalysis Tutorials](https://www.mdanalysis.org/MDAnalysisTutorial/)

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Describir los principios físicos de la dinámica molecular
- ✅ Implementar el potencial de Lennard-Jones y calcular fuerzas
- ✅ Generar velocidades iniciales con la distribución de Maxwell-Boltzmann
- ✅ Distinguir entre los ensambles NVE, NVT y NPT
- ✅ Ejecutar una simulación sencilla de partículas con Python

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 5.1: Fundamentos de Dinámica Molecular**

[![Siguiente](https://img.shields.io/badge/Actividad_5.2_➡️-Integradores_y_Algoritmos-green.svg)](02_integradores_algoritmos.ipynb)

---

📚 **[Volver al Módulo 5](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G*

</div>